# Scraper de precios capilares de Cromantic – versión Colab

Evidencia de aprendizaje 2 · Programación para Análisis de Datos · IU Digital de Antioquia

Este notebook arma el mismo proyecto del repositorio `cromantic-scraper-devops` y deja:

1. Verificar el patrón de paginación real de Cromantic.
2. Correr las pruebas (pytest) y la revisión de estilo (flake8), igual que el pipeline de CI.
3. Ejecutar el scraper, ver el análisis y el reporte HTML.
4. Descargar el CSV y el proyecto completo en .zip para subirlo a GitHub.

> Colab no ejecuta GitHub Actions. La integración y el despliegue continuos ocurren en GitHub; aquí solo se reproduce lo que hace el pipeline.

Ejecuta las celdas en orden: **Entorno de ejecución → Ejecutar todas**.

## 1. Instalación de herramientas de prueba

`requests`, `beautifulsoup4` y `pandas` ya vienen en Colab. Se agregan `pytest`, `pytest-cov` y `flake8`.

In [ ]:
!pip install -q pytest pytest-cov flake8

## 2. Creación de la estructura del proyecto

Se crean las carpetas igual que en el repositorio y Colab queda trabajando dentro de la carpeta del proyecto.

In [ ]:
import os

PROYECTO = '/content/cromantic-scraper-devops'
for carpeta in ['src/cromantic_scraper', 'tests/fixtures', '.github/workflows', 'data', 'site']:
    os.makedirs(os.path.join(PROYECTO, carpeta), exist_ok=True)
os.chdir(PROYECTO)
print('Trabajando en:', os.getcwd())

## 3. Código fuente

Cada celda `%%writefile` guarda un archivo del proyecto. Si cambias algo aquí, cambia también en el repositorio.

### 3.1 `__init__.py`

In [ ]:
%%writefile src/cromantic_scraper/__init__.py
"""Scraper de precios capilares de Cromantic."""

__version__ = "1.0.0"


### 3.2 `scraper.py` – petición, lectura de JSON-LD y paginación

Revisa `URL_PAGINACION` después de correr la sección 5.

In [ ]:
%%writefile src/cromantic_scraper/scraper.py
"""Extracción de productos de Cromantic a partir de sus bloques JSON-LD.

Las funciones de lectura (parse_*) reciben HTML como texto y no hacen
peticiones de red. Así se pueden probar en el pipeline de CI con archivos
de ejemplo, sin depender de que el sitio esté disponible.
"""

import html
import json
import logging
import time

import requests
from bs4 import BeautifulSoup

logger = logging.getLogger(__name__)

BASE_URL = "https://www.cromantic.com/"
# Patrón de paginación. Verificar en el navegador cómo numera el sitio
# sus páginas (por ejemplo ?page=2) antes de cambiarlo.
URL_PAGINACION = "https://www.cromantic.com/capilar?page={pagina}"
MARCA_POR_DEFECTO = "Cromantic"
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
}


def obtener_marca(producto):
    """Devuelve el nombre de la marca, venga como diccionario o como texto."""
    marca = producto.get("brand", {})
    if isinstance(marca, dict):
        return marca.get("name") or MARCA_POR_DEFECTO
    if isinstance(marca, str) and marca.strip():
        return marca.strip()
    return MARCA_POR_DEFECTO


def obtener_precio(producto):
    """Devuelve el precio como float, o None si no está disponible."""
    ofertas = producto.get("offers", {})
    if isinstance(ofertas, list) and ofertas:
        ofertas = ofertas[0]
    if isinstance(ofertas, dict):
        valor = (
            ofertas.get("lowPrice")
            or ofertas.get("price")
            or ofertas.get("highPrice")
        )
        if valor not in (None, ""):
            try:
                return float(valor)
            except (TypeError, ValueError):
                return None
    return None


def formatear_cop(precio):
    """Convierte 49900.0 en '$49.900'."""
    if precio is None:
        return "No disponible"
    return f"${precio:,.0f}".replace(",", ".")


def armar_registro(producto, categoria):
    """Construye el diccionario de un producto con las columnas finales."""
    nombre = html.unescape(producto.get("name", "No disponible"))
    sku = str(producto.get("sku") or producto.get("@id") or "No disponible")
    marca = html.unescape(obtener_marca(producto))
    precio = obtener_precio(producto)
    return {
        "sku": sku,
        "marca": marca,
        "nombre": nombre,
        "precio_cop": precio,
        "precio_texto": formatear_cop(precio),
        "categoria": categoria,
    }


def parse_productos(contenido_html, categoria="Belleza"):
    """Lee los bloques JSON-LD (ItemList o Product) de una página."""
    soup = BeautifulSoup(contenido_html, "html.parser")
    productos = []
    for script in soup.find_all("script", type="application/ld+json"):
        if not script.string:
            continue
        try:
            datos = json.loads(script.string)
        except json.JSONDecodeError as error:
            logger.warning("Bloque JSON-LD inválido: %s", error)
            continue
        bloques = datos if isinstance(datos, list) else [datos]
        for bloque in bloques:
            if not isinstance(bloque, dict):
                continue
            tipo = bloque.get("@type")
            if tipo == "ItemList":
                for envoltura in bloque.get("itemListElement", []):
                    producto = envoltura.get("item", {})
                    if producto:
                        productos.append(armar_registro(producto, categoria))
            elif tipo == "Product":
                productos.append(armar_registro(bloque, categoria))
    return productos


def descargar(url, sesion=None, timeout=10):
    """Hace la petición GET y devuelve el HTML, o None si falla."""
    cliente = sesion or requests
    try:
        respuesta = cliente.get(url, headers=HEADERS, timeout=timeout)
    except requests.RequestException as error:
        logger.error("Error de red en %s: %s", url, error)
        return None
    if respuesta.status_code != 200:
        logger.warning("%s respondió %s", url, respuesta.status_code)
        return None
    return respuesta.text


def scrape_paginado(paginas=12, patron=URL_PAGINACION, pausa=1.5,
                    categoria="Capilar", obtener_html=descargar):
    """Recorre las páginas de una categoría y consolida productos únicos.

    Se detiene en cuanto una página no aporta SKU nuevos: eso indica que el
    sitio está devolviendo la misma página o que se acabó el catálogo.
    """
    vistos = set()
    resultado = []
    for pagina in range(1, paginas + 1):
        contenido = obtener_html(patron.format(pagina=pagina))
        if contenido is None:
            continue
        nuevos = [p for p in parse_productos(contenido, categoria)
                  if p["sku"] not in vistos]
        logger.info("Página %s: %s productos nuevos", pagina, len(nuevos))
        if not nuevos:
            logger.info("Sin productos nuevos; fin de la paginación.")
            break
        for producto in nuevos:
            vistos.add(producto["sku"])
            resultado.append(producto)
        if pausa:
            time.sleep(pausa)
    return resultado


### 3.3 `analysis.py` – estadísticas con pandas

In [ ]:
%%writefile src/cromantic_scraper/analysis.py
"""Análisis descriptivo de los productos extraídos."""

import pandas as pd

COLUMNAS = ["sku", "marca", "nombre", "precio_cop", "precio_texto",
            "categoria"]


def a_dataframe(productos):
    """Convierte la lista de productos en DataFrame sin SKU repetidos."""
    df = pd.DataFrame(productos, columns=COLUMNAS)
    return df.drop_duplicates(subset=["sku"]).reset_index(drop=True)


def resumen_precios(df):
    """Estadísticas básicas de precio (COP), ignorando precios vacíos."""
    precios = df["precio_cop"].dropna()
    if precios.empty:
        return {"productos": 0, "promedio": None, "mediana": None,
                "minimo": None, "maximo": None}
    return {
        "productos": int(precios.count()),
        "promedio": round(float(precios.mean())),
        "mediana": round(float(precios.median())),
        "minimo": round(float(precios.min())),
        "maximo": round(float(precios.max())),
    }


def por_marca(df):
    """Cantidad de productos y precio promedio por marca."""
    tabla = (
        df.groupby("marca")["precio_cop"]
        .agg(productos="count", precio_promedio="mean")
        .sort_values("precio_promedio", ascending=False)
        .round(0)
        .reset_index()
    )
    return tabla


def mas_caros(df, n=5):
    """Los n productos de mayor precio."""
    return df.nlargest(n, "precio_cop")[["marca", "nombre", "precio_texto"]]


### 3.4 `report.py` – página HTML del reporte

In [ ]:
%%writefile src/cromantic_scraper/report.py
"""Genera la página HTML que se publica en GitHub Pages."""

import html
from datetime import datetime, timezone

from .analysis import mas_caros, por_marca, resumen_precios
from .scraper import formatear_cop

PLANTILLA = """<!doctype html>
<html lang="es">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Precios capilares Cromantic</title>
<style>
body {{ font-family: system-ui, sans-serif; max-width: 880px;
       margin: 0 auto; padding: 24px 16px; color: #1d1d1f; }}
h1 {{ font-size: 1.6rem; margin-bottom: 4px; }}
.meta {{ color: #666; font-size: .9rem; margin-bottom: 24px; }}
.kpis {{ display: grid; grid-template-columns: repeat(auto-fit,
        minmax(150px, 1fr)); gap: 12px; margin-bottom: 28px; }}
.kpi {{ border: 1px solid #ddd; border-radius: 8px; padding: 12px; }}
.kpi b {{ display: block; font-size: 1.3rem; }}
table {{ border-collapse: collapse; width: 100%; margin-bottom: 28px;
        font-size: .92rem; }}
th, td {{ border-bottom: 1px solid #e5e5e5; padding: 8px 6px;
         text-align: left; }}
th {{ background: #f6f6f6; }}
td.num {{ text-align: right; white-space: nowrap; }}
</style>
</head>
<body>
<h1>Precios de productos capilares en Cromantic</h1>
<p class="meta">Actualizado: {fecha} &middot; Fuente: {fuente}</p>
<div class="kpis">
<div class="kpi">Productos<b>{productos}</b></div>
<div class="kpi">Precio promedio<b>{promedio}</b></div>
<div class="kpi">Mediana<b>{mediana}</b></div>
<div class="kpi">Rango<b>{minimo} &ndash; {maximo}</b></div>
</div>
<h2>Precio promedio por marca</h2>
<table><tr><th>Marca</th><th>Productos</th><th>Precio promedio</th></tr>
{filas_marca}
</table>
<h2>Los 5 productos más caros</h2>
<table><tr><th>Marca</th><th>Producto</th><th>Precio</th></tr>
{filas_caros}
</table>
<h2>Catálogo extraído</h2>
<table><tr><th>SKU</th><th>Marca</th><th>Producto</th><th>Precio</th></tr>
{filas_todos}
</table>
</body>
</html>
"""


def _e(texto):
    return html.escape(str(texto))


def construir_html(df, fuente="https://www.cromantic.com/capilar"):
    """Devuelve el HTML del reporte a partir del DataFrame."""
    r = resumen_precios(df)
    filas_marca = "\n".join(
        f"<tr><td>{_e(f.marca)}</td><td class='num'>{f.productos}</td>"
        f"<td class='num'>{formatear_cop(f.precio_promedio)}</td></tr>"
        for f in por_marca(df).itertuples()
    )
    filas_caros = "\n".join(
        f"<tr><td>{_e(f.marca)}</td><td>{_e(f.nombre)}</td>"
        f"<td class='num'>{_e(f.precio_texto)}</td></tr>"
        for f in mas_caros(df).itertuples()
    )
    filas_todos = "\n".join(
        f"<tr><td>{_e(f.sku)}</td><td>{_e(f.marca)}</td>"
        f"<td>{_e(f.nombre)}</td><td class='num'>{_e(f.precio_texto)}</td></tr>"
        for f in df.sort_values("precio_cop", ascending=False).itertuples()
    )
    return PLANTILLA.format(
        fecha=datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC"),
        fuente=_e(fuente),
        productos=r["productos"],
        promedio=formatear_cop(r["promedio"]),
        mediana=formatear_cop(r["mediana"]),
        minimo=formatear_cop(r["minimo"]),
        maximo=formatear_cop(r["maximo"]),
        filas_marca=filas_marca,
        filas_caros=filas_caros,
        filas_todos=filas_todos,
    )


### 3.5 `__main__.py` – scraping → CSV → HTML, con respaldo

In [ ]:
%%writefile src/cromantic_scraper/__main__.py
"""Punto de entrada: python -m cromantic_scraper

Ejecuta el scraping, guarda el CSV y genera la página del reporte.
Si el sitio no responde (por ejemplo, bloquea la IP del runner de GitHub),
usa como respaldo el último CSV guardado en el repositorio, para que el
despliegue no publique una página vacía.
"""

import argparse
import logging
import sys
from pathlib import Path

import pandas as pd

from .analysis import a_dataframe
from .report import construir_html
from .scraper import scrape_paginado


def cargar_csv(ruta):
    """Lee el CSV del proyecto o el que exportaba el notebook de Colab."""
    df = pd.read_csv(ruta, sep=";", encoding="utf-8-sig", dtype=str)
    if "Referencia (SKU)" in df.columns:  # formato del notebook original
        partes = df["Nombre/Marca"].str.extract(r"^\[(.*?)\]\s*(.*)$")
        df = pd.DataFrame({
            "sku": df["Referencia (SKU)"],
            "marca": partes[0],
            "nombre": partes[1],
            "precio_cop": df["Valor (COP)"],
            "precio_texto": df["Valor"],
            "categoria": df["Características"],
        })
    df["precio_cop"] = pd.to_numeric(df["precio_cop"], errors="coerce")
    return df


def main(argv=None):
    parser = argparse.ArgumentParser(description="Scraper capilar Cromantic")
    parser.add_argument("--paginas", type=int, default=12)
    parser.add_argument("--csv", default="data/productos_cromantic.csv")
    parser.add_argument("--salida", default="site/index.html")
    args = parser.parse_args(argv)

    logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
    log = logging.getLogger("cromantic")

    ruta_csv = Path(args.csv)
    productos = scrape_paginado(paginas=args.paginas)

    if productos:
        df = a_dataframe(productos)
        ruta_csv.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(ruta_csv, index=False, encoding="utf-8-sig", sep=";")
        log.info("Guardados %s productos en %s", len(df), ruta_csv)
    elif ruta_csv.exists():
        log.warning("Sin datos nuevos; se usa el CSV existente %s", ruta_csv)
        df = cargar_csv(ruta_csv)
    else:
        log.error("No hay datos para publicar.")
        return 1

    ruta_html = Path(args.salida)
    ruta_html.parent.mkdir(parents=True, exist_ok=True)
    ruta_html.write_text(construir_html(df), encoding="utf-8")
    log.info("Reporte generado en %s", ruta_html)
    return 0


if __name__ == "__main__":
    sys.exit(main())


## 4. Pruebas y configuración

### 4.1 Página de ejemplo para las pruebas (sin internet)

In [ ]:
%%writefile tests/fixtures/capilar_pagina1.html
<!doctype html>
<html lang="es">
<head>
<title>Capilar | Cromantic (página de ejemplo para pruebas)</title>
<script type="application/ld+json">
{
  "@context": "https://schema.org",
  "@type": "ItemList",
  "itemListElement": [
    {"@type": "ListItem", "position": 1, "item": {
      "@type": "Product", "sku": "11030",
      "name": "Kit Olaplex Hair Hydration Heroes",
      "brand": {"@type": "Brand", "name": "Olaplex"},
      "offers": {"@type": "AggregateOffer", "lowPrice": 165990, "highPrice": 165990, "priceCurrency": "COP"}}},
    {"@type": "ListItem", "position": 2, "item": {
      "@type": "Product", "sku": "11015",
      "name": "Shampoo Everyday Love Miracle 300ml",
      "brand": "Everyday Love",
      "offers": {"@type": "Offer", "price": "49900", "priceCurrency": "COP"}}},
    {"@type": "ListItem", "position": 3, "item": {
      "@type": "Product", "sku": "10996",
      "name": "T&oacute;nico Romero Anyeluz 110ml",
      "brand": {"@type": "Brand", "name": "Anyeluz"},
      "offers": {"@type": "Offer", "price": "44990", "priceCurrency": "COP"}}}
  ]
}
</script>
<script type="application/ld+json">{"@context": "https://schema.org", "@type": "Organization", "name": "Cromantic"}</script>
<script type="application/ld+json">{ esto no es JSON válido </script>
</head>
<body><h1>Capilar</h1></body>
</html>


### 4.2 `test_scraper.py`

In [ ]:
%%writefile tests/test_scraper.py
"""Pruebas unitarias del scraper. No hacen peticiones reales a internet."""

from pathlib import Path

import pytest

from cromantic_scraper import scraper

FIXTURES = Path(__file__).parent / "fixtures"


@pytest.fixture
def html_capilar():
    return (FIXTURES / "capilar_pagina1.html").read_text(encoding="utf-8")


def test_parse_lee_los_tres_productos_del_itemlist(html_capilar):
    productos = scraper.parse_productos(html_capilar, "Capilar")
    assert [p["sku"] for p in productos] == ["11030", "11015", "10996"]


def test_parse_ignora_json_invalido_y_tipos_ajenos(html_capilar):
    # El fixture trae un bloque Organization y uno roto: no deben romper nada
    productos = scraper.parse_productos(html_capilar)
    assert len(productos) == 3


def test_marca_como_dict_y_como_texto(html_capilar):
    marcas = [p["marca"] for p in scraper.parse_productos(html_capilar)]
    assert marcas == ["Olaplex", "Everyday Love", "Anyeluz"]


def test_marca_por_defecto_cuando_falta():
    assert scraper.obtener_marca({}) == "Cromantic"
    assert scraper.obtener_marca({"brand": "  "}) == "Cromantic"


def test_precio_numerico_desde_offer_y_aggregateoffer(html_capilar):
    precios = [p["precio_cop"] for p in scraper.parse_productos(html_capilar)]
    assert precios == [165990.0, 49900.0, 44990.0]


def test_precio_invalido_devuelve_none():
    assert scraper.obtener_precio({"offers": {"price": "consultar"}}) is None
    assert scraper.obtener_precio({}) is None


def test_formato_pesos_colombianos():
    assert scraper.formatear_cop(1499900.0) == "$1.499.900"
    assert scraper.formatear_cop(None) == "No disponible"


def test_corrige_entidades_html(html_capilar):
    nombres = [p["nombre"] for p in scraper.parse_productos(html_capilar)]
    assert "Tónico Romero Anyeluz 110ml" in nombres


def test_paginacion_se_detiene_si_la_pagina_se_repite(html_capilar):
    """Reproduce el error del notebook: 12 páginas que devuelven lo mismo."""
    urls = []

    def falso_get(url):
        urls.append(url)
        return html_capilar

    productos = scraper.scrape_paginado(paginas=12, pausa=0,
                                        obtener_html=falso_get)
    assert len(productos) == 3
    assert len(urls) == 2  # la página 2 no trae nada nuevo y corta


def test_paginacion_salta_paginas_caidas(html_capilar):
    respuestas = iter([None, html_capilar, None])

    productos = scraper.scrape_paginado(
        paginas=3, pausa=0, obtener_html=lambda url: next(respuestas))
    assert len(productos) == 3


### 4.3 `test_analysis.py`

In [ ]:
%%writefile tests/test_analysis.py
"""Pruebas del análisis descriptivo y del reporte HTML."""

from pathlib import Path

from cromantic_scraper import analysis, report, scraper
from cromantic_scraper.__main__ import cargar_csv

FIXTURES = Path(__file__).parent / "fixtures"


def productos_ejemplo():
    html = (FIXTURES / "capilar_pagina1.html").read_text(encoding="utf-8")
    return scraper.parse_productos(html, "Capilar")


def test_dataframe_elimina_sku_repetidos():
    datos = productos_ejemplo() + productos_ejemplo()
    assert len(analysis.a_dataframe(datos)) == 3


def test_resumen_de_precios():
    df = analysis.a_dataframe(productos_ejemplo())
    r = analysis.resumen_precios(df)
    assert r["productos"] == 3
    assert r["minimo"] == 44990
    assert r["maximo"] == 165990
    assert r["mediana"] == 49900


def test_resumen_vacio_no_falla():
    df = analysis.a_dataframe([])
    assert analysis.resumen_precios(df)["productos"] == 0


def test_marca_mas_cara_primero():
    df = analysis.a_dataframe(productos_ejemplo())
    assert analysis.por_marca(df).iloc[0]["marca"] == "Olaplex"


def test_reporte_html_contiene_datos_y_escapa_texto():
    datos = productos_ejemplo()
    datos[0]["nombre"] = "<script>alert(1)</script>"
    pagina = report.construir_html(analysis.a_dataframe(datos))
    assert "$165.990" in pagina
    assert "<script>alert(1)</script>" not in pagina


def test_lee_csv_del_notebook_original(tmp_path):
    ruta = tmp_path / "viejo.csv"
    ruta.write_text(
        "Referencia (SKU);Nombre/Marca;Valor (COP);Valor;Características\n"
        "11030;[Olaplex] Kit Olaplex Hair Hydration Heroes;165990.0;"
        "$165.990;Capilar, Belleza\n",
        encoding="utf-8-sig",
    )
    df = cargar_csv(ruta)
    assert df.loc[0, "marca"] == "Olaplex"
    assert df.loc[0, "precio_cop"] == 165990.0


### `pyproject.toml`

In [ ]:
%%writefile pyproject.toml
[build-system]
requires = ["setuptools>=61"]
build-backend = "setuptools.build_meta"

[project]
name = "cromantic-scraper"
version = "1.0.0"
description = "Scraper de precios capilares de Cromantic con flujo DevOps"
requires-python = ">=3.10"
dependencies = ["requests>=2.31", "beautifulsoup4>=4.12", "pandas>=2.0"]

[tool.setuptools.packages.find]
where = ["src"]

[tool.pytest.ini_options]
testpaths = ["tests"]
pythonpath = ["src"]


### `requirements.txt`

In [ ]:
%%writefile requirements.txt
requests>=2.31
beautifulsoup4>=4.12
pandas>=2.0


### `requirements-dev.txt`

In [ ]:
%%writefile requirements-dev.txt
-r requirements.txt
pytest>=8.0
pytest-cov>=5.0
flake8>=7.0


### `.flake8`

In [ ]:
%%writefile .flake8
[flake8]
max-line-length = 88
exclude = .git,__pycache__,notebooks,site,build


### `.gitignore`

In [ ]:
%%writefile .gitignore
__pycache__/
*.pyc
.pytest_cache/
.coverage
htmlcov/
site/
*.egg-info/
.venv/
.ipynb_checkpoints/


### `Dockerfile`

In [ ]:
%%writefile Dockerfile
# Imagen para ejecutar el scraper igual en cualquier máquina
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt pyproject.toml ./
COPY src ./src
RUN pip install --no-cache-dir -r requirements.txt && pip install --no-cache-dir .
COPY data ./data
CMD ["python", "-m", "cromantic_scraper"]


### `README.md`

In [ ]:
%%writefile README.md
# Scraper de precios capilares de Cromantic con flujo DevOps

Proyecto de la Evidencia de aprendizaje 2 (Programación para Análisis de Datos, IU Digital de Antioquia).
Extrae SKU, marca, nombre y precio de la categoría capilar de [Cromantic](https://www.cromantic.com/),
los guarda en CSV y publica un reporte en GitHub Pages. Cada cambio pasa por pruebas automáticas antes de llegar a `main`.

![CI](https://github.com/AndreaAramburo93/cromantic-scraper-devops/actions/workflows/ci.yml/badge.svg)

## Estructura

```
cromantic-scraper-devops/
├── src/cromantic_scraper/
│   ├── scraper.py      # petición GET, lectura de JSON-LD y paginación
│   ├── analysis.py     # estadísticas con pandas
│   ├── report.py       # página HTML del reporte
│   └── __main__.py     # punto de entrada: scraping → CSV → HTML
├── tests/              # pruebas unitarias (pytest) con HTML de ejemplo, sin red
├── data/               # productos_cromantic.csv (respaldo y versión de los datos)
├── notebooks/          # notebook original de la Evidencia 1 (Colab)
├── .github/workflows/
│   ├── ci.yml          # integración continua: flake8 + pytest en cada push y PR
│   └── deploy.yml      # despliegue continuo: scraper + GitHub Pages al llegar a main
├── Dockerfile
├── pyproject.toml
└── requirements*.txt
```

## Ejecutar en local

```bash
python -m venv .venv
source .venv/bin/activate          # Windows: .venv\Scripts\activate
pip install -r requirements-dev.txt
pip install -e .

python -m cromantic_scraper --paginas 12   # genera data/productos_cromantic.csv y site/index.html
pytest                                      # corre las pruebas
flake8 src tests                            # revisa estilo PEP 8
```

Con Docker:

```bash
docker build -t cromantic-scraper .
docker run --rm -v "$PWD/data:/app/data" -v "$PWD/site:/app/site" cromantic-scraper
```

## Flujo de trabajo DevOps

Se usa **GitHub Flow**: `main` siempre está estable y todo cambio entra por una rama y un pull request.

1. Se crea una rama desde `main` (`feature/...`, `fix/...`, `docs/...`).
2. Se hacen commits pequeños con mensajes claros (`feat:`, `fix:`, `test:`, `docs:`, `ci:`).
3. Cada `push` dispara **CI** (`ci.yml`): instala dependencias, corre `flake8` y `pytest` en Python 3.10, 3.11 y 3.12.
4. Se abre un pull request hacia `main`. La regla de protección exige que CI esté en verde y una revisión.
5. Al fusionar, se dispara **CD** (`deploy.yml`): repite las pruebas, ejecuta el scraper, guarda el CSV
   actualizado y publica el reporte en GitHub Pages. Al final hace una prueba de humo sobre la URL publicada.
6. CD también corre cada lunes a las 7:00 a. m. (hora Colombia) para refrescar precios.

Si Cromantic no responde desde el servidor de GitHub, el scraper usa el último CSV guardado,
así el sitio nunca queda vacío.

## Configuración en GitHub (una sola vez)

1. Crear el repositorio `cromantic-scraper-devops` (público) y subir el código.
2. **Settings → Pages → Source: GitHub Actions.**
3. **Settings → Branches → Add rule** para `main`: *Require a pull request before merging* y
   *Require status checks to pass* (seleccionar los trabajos `pruebas`).
4. **Settings → Actions → General → Workflow permissions: Read and write.**
5. Copiar en `data/` el `productos_cromantic.csv` descargado de Colab como primer respaldo.

## Comandos de Git usados en el proyecto

```bash
# 1. Repositorio y primer push
git init
git add .
git commit -m "chore: estructura inicial del proyecto"
git branch -M main
git remote add origin https://github.com/AndreaAramburo93/cromantic-scraper-devops.git
git push -u origin main

# 2. Rama para el scraper
git checkout -b feature/scraper
git add src/cromantic_scraper/scraper.py
git commit -m "feat: extracción de productos desde JSON-LD"
git push -u origin feature/scraper      # luego: pull request en GitHub

# 3. Simular un error y revertirlo (rollback)
git commit -am "feat: cambia patrón de paginación"   # commit con error
git log --oneline                                    # ubicar el commit
git revert HEAD                                      # crea un commit que deshace el anterior
git push

# 4. Segunda rama (otro integrante): reporte y visualización
git checkout main && git pull
git checkout -b feature/reporte
git add src/cromantic_scraper/report.py
git commit -m "feat: reporte HTML para GitHub Pages"
git push -u origin feature/reporte

# 5. Resolver un conflicto al actualizar la rama con main
git fetch origin
git merge origin/main        # si hay conflicto, editar el archivo marcado <<<<<<< ======= >>>>>>>
git add archivo_resuelto.py
git commit -m "fix: resuelve conflicto con main"
git push

# 6. Versión estable
git checkout main && git pull
git tag -a v1.0.0 -m "Primera versión estable"
git push origin v1.0.0
```

## Pruebas automatizadas

| Archivo | Qué valida |
| --- | --- |
| `tests/test_scraper.py` | Lectura de ItemList, marca como texto o diccionario, precio en `Offer` y `AggregateOffer`, entidades HTML, JSON roto, corte de paginación cuando una página se repite |
| `tests/test_analysis.py` | Eliminación de SKU repetidos, estadísticas de precio, orden por marca, escape de HTML en el reporte, lectura del CSV del notebook |

Las pruebas usan `tests/fixtures/capilar_pagina1.html`, una página de ejemplo con la misma
estructura JSON-LD del sitio. No dependen de internet, por eso no fallan si la tienda cambia o se cae.

## Uso responsable

El scraper espera 1,5 s entre páginas, se identifica con un User-Agent de navegador y solo lee
datos públicos de precio. Antes de ampliar el alcance hay que revisar `https://www.cromantic.com/robots.txt`
y los términos del sitio.


### `.github/workflows/ci.yml`

In [ ]:
%%writefile .github/workflows/ci.yml
# Integración continua: se ejecuta en cada push y en cada pull request.
# Si el estilo (flake8) o alguna prueba (pytest) falla, el commit queda en rojo
# y la regla de protección de main impide fusionar el pull request.
name: CI

on:
  push:
    branches: ["**"]
  pull_request:
    branches: [main]

jobs:
  pruebas:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python-version: ["3.10", "3.11", "3.12"]
    steps:
      - name: Descargar el código
        uses: actions/checkout@v4

      - name: Instalar Python ${{ matrix.python-version }}
        uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}
          cache: pip

      - name: Instalar dependencias
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements-dev.txt
          pip install -e .

      - name: Revisar estilo PEP 8 (flake8)
        run: flake8 src tests

      - name: Ejecutar pruebas unitarias con cobertura
        run: pytest --cov=cromantic_scraper --cov-report=term-missing


### `.github/workflows/deploy.yml`

In [ ]:
%%writefile .github/workflows/deploy.yml
# Despliegue continuo: cuando un cambio llega a main (merge de un pull
# request), se vuelven a correr las pruebas, se ejecuta el scraper y se
# publica el reporte en GitHub Pages. También corre cada lunes para
# refrescar los precios, y se puede lanzar a mano desde la pestaña Actions.
name: CD - Publicar reporte

on:
  push:
    branches: [main]
  schedule:
    - cron: "0 12 * * 1"   # lunes 7:00 a. m. hora Colombia
  workflow_dispatch:

permissions:
  contents: write   # para guardar el CSV actualizado
  pages: write
  id-token: write

concurrency:
  group: pages
  cancel-in-progress: false

jobs:
  construir:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
          cache: pip

      - name: Instalar dependencias
        run: |
          pip install -r requirements-dev.txt
          pip install -e .

      - name: Pruebas antes de desplegar
        run: pytest -q

      - name: Ejecutar el scraper y generar el reporte
        run: python -m cromantic_scraper --paginas 12

      - name: Guardar el CSV actualizado en el repositorio
        run: |
          git config user.name "github-actions[bot]"
          git config user.email "github-actions[bot]@users.noreply.github.com"
          git add data/productos_cromantic.csv
          git diff --cached --quiet || git commit -m "data: actualiza precios [skip ci]"
          git push || echo "Sin cambios para subir"

      - name: Preparar artefacto de Pages
        uses: actions/upload-pages-artifact@v3
        with:
          path: site

  desplegar:
    needs: construir
    runs-on: ubuntu-latest
    environment:
      name: github-pages
      url: ${{ steps.deployment.outputs.page_url }}
    steps:
      - name: Publicar en GitHub Pages
        id: deployment
        uses: actions/deploy-pages@v4

      - name: Prueba de humo sobre el sitio publicado
        run: |
          sleep 15
          curl --fail --silent "${{ steps.deployment.outputs.page_url }}" \
            | grep -q "Precios de productos capilares"


## 5. Verificar el patrón de paginación

Esta celda descarga las páginas 1 y 2 con cada patrón candidato y compara los SKU.
El patrón correcto es el que trae **SKU distintos** en la página 2. Si ninguno funciona,
abre la categoría Capilar en el navegador, pasa a la página 2 y copia cómo cambia la URL.

In [ ]:
import sys
sys.path.insert(0, os.path.join(PROYECTO, 'src'))

import importlib
import cromantic_scraper.scraper as scraper
importlib.reload(scraper)

candidatos = [
    'https://www.cromantic.com/capilar?page={pagina}',
    'https://www.cromantic.com/capilar?p={pagina}',
    'https://www.cromantic.com/capilar/page/{pagina}',
    'https://www.cromantic.com/capilar{pagina}',  # el patrón del notebook original
]

for patron in candidatos:
    skus = []
    for pagina in (1, 2):
        contenido = scraper.descargar(patron.format(pagina=pagina))
        productos = scraper.parse_productos(contenido, 'Capilar') if contenido else []
        skus.append({p['sku'] for p in productos})
    nuevos = len(skus[1] - skus[0])
    estado = 'FUNCIONA' if skus[1] and nuevos else 'no sirve'
    print(f'{estado:9} | pág1={len(skus[0]):3} pág2={len(skus[1]):3} SKU nuevos en pág2={nuevos:3} | {patron}')

In [ ]:
# Escribe aquí el patrón que salió como FUNCIONA y ejecuta la celda.
# Actualiza scraper.py en este notebook (y en el repositorio) para que el resto use ese patrón.
PATRON_CORRECTO = 'https://www.cromantic.com/capilar?page={pagina}'

ruta = 'src/cromantic_scraper/scraper.py'
texto = open(ruta, encoding='utf-8').read()
inicio = texto.index('URL_PAGINACION = ')
fin = texto.index('\n', inicio)
texto = texto[:inicio] + f'URL_PAGINACION = "{PATRON_CORRECTO}"' + texto[fin:]
open(ruta, 'w', encoding='utf-8').write(texto)
importlib.reload(scraper)
print('Patrón en uso:', scraper.URL_PAGINACION)

## 6. Lo mismo que hace el pipeline de CI

Revisión de estilo PEP 8 y pruebas unitarias con cobertura. Si algo sale en rojo aquí, también saldría en rojo en GitHub Actions.

In [ ]:
!flake8 src tests && echo 'flake8: sin errores de estilo'

In [ ]:
!python -m pytest --cov=cromantic_scraper --cov-report=term-missing

## 7. Ejecutar el scraper

Opcional: si tienes el CSV de la Evidencia 1, súbelo en la celda siguiente. El scraper lo usa como respaldo si la tienda no responde.

In [ ]:
# Opcional: subir el productos_cromantic.csv de la Evidencia 1 como respaldo
from google.colab import files
subidos = files.upload()
for nombre in subidos:
    os.replace(nombre, 'data/productos_cromantic.csv')
    print('Respaldo guardado en data/productos_cromantic.csv')

In [ ]:
!PYTHONPATH=src python -m cromantic_scraper --paginas 12

## 8. Análisis descriptivo

In [ ]:
import cromantic_scraper.analysis as analysis
from cromantic_scraper.__main__ import cargar_csv
importlib.reload(analysis)

if not os.path.exists('data/productos_cromantic.csv'):
    raise SystemExit('No hay datos: la tienda no respondió y no se subió un CSV de respaldo (sección 7).')

df = cargar_csv('data/productos_cromantic.csv')
print(f'Productos únicos: {len(df)}\n')
print('Resumen de precios (COP):', analysis.resumen_precios(df))
display(analysis.por_marca(df))
display(analysis.mas_caros(df))

In [ ]:
import matplotlib.pyplot as plt

if 'df' not in globals():
    raise SystemExit('Primero ejecuta la celda anterior con datos.')

marcas = analysis.por_marca(df).sort_values('precio_promedio')
fig, ax = plt.subplots(figsize=(8, max(3, 0.4 * len(marcas))))
ax.barh(marcas['marca'], marcas['precio_promedio'], color='#4C78A8')
ax.set_xlabel('Precio promedio (COP)')
ax.set_title('Precio promedio por marca – categoría Capilar')
ax.xaxis.set_major_formatter(lambda x, _: f'${x:,.0f}'.replace(',', '.'))
for lado in ('top', 'right'):
    ax.spines[lado].set_visible(False)
plt.tight_layout()
plt.show()

## 9. Reporte HTML (lo que se publica en GitHub Pages)

In [ ]:
from IPython.display import HTML, display
if os.path.exists('site/index.html'):
    display(HTML(open('site/index.html', encoding='utf-8').read()))
else:
    print('No se generó el reporte: revisa la salida de la sección 7.')

## 10. Descargas

El CSV con los precios y el proyecto completo en .zip, listo para subirlo a GitHub.

In [ ]:
import shutil
from google.colab import files

shutil.rmtree('site', ignore_errors=True)
os.makedirs('site', exist_ok=True)
for basura in ['.pytest_cache', 'src/cromantic_scraper/__pycache__', 'tests/__pycache__']:
    shutil.rmtree(basura, ignore_errors=True)
if os.path.exists('.coverage'):
    os.remove('.coverage')

zip_ruta = shutil.make_archive('/content/cromantic-scraper-devops', 'zip', '/content', 'cromantic-scraper-devops')
if os.path.exists('data/productos_cromantic.csv'):
    files.download('data/productos_cromantic.csv')
files.download(zip_ruta)

## 11. Siguiente paso: subirlo a GitHub

1. Descomprime el .zip en tu computador.
2. Sigue la sección **Comandos de Git usados en el proyecto** del `README.md` (rama, commits, pull request, revert, tag).
3. En GitHub: **Settings → Pages → Source: GitHub Actions** y **Settings → Actions → General → Read and write**.
4. Cada push dispara `ci.yml`; cada merge a `main` dispara `deploy.yml` y publica el reporte.